# 4. Deployment Strategy - Interactive Phishing Detection Demo

Building a production-ready demonstration of our hybrid phishing detection system using Streamlit.

## 1. Deployment Overview

**What we're building:**

An interactive phishing detection demo using Streamlit that demonstrates our hybrid approach on dataset4 URLs.

**Demo Mode approach:**
- Select URLs from dataset4 (pre-extracted features)
- Run through hybrid system: Rules → XGBoost
- Show prediction vs actual label
- Display feature breakdown and explanation

**Why Demo Mode (not Live URL scanning)?**

**Live Mode would require:**
1. Fetching live webpages (`requests.get(url)`)
2. Extracting ALL 49 features from raw HTML:
   - Easy features: URLLength, IsHTTPS, NoOfJS (count `<script>` tags)
   - Hard features: TLDLegitimateProb (need TLD reputation database), CharContinuationRate (complex calculation), DomainTitleMatchScore (fuzzy matching)
3. Handling failures: Timeouts, CAPTCHAs, blocked requests, malformed HTML
4. Building 49 feature extractors (significant engineering effort)

**Why we chose Demo Mode:**
- **Focus on ML, not web scraping**: This project demonstrates machine learning for phishing detection, not production web crawling
- **Reliability**: Demo always works (no network issues, timeouts, blocked requests)
- **Assessment-friendly**: Reproducible results for grading
- **Proof of concept**: Shows model performance without production engineering complexity

**In production:** Feature extraction would be handled by a dedicated service (similar to Google Safe Browsing's web crawlers). Our model consumes features, not raw URLs.

**Tech stack:**
- Streamlit (interactive UI)
- XGBoost (pre-trained model from notebook 3)
- pandas (feature handling)

## 2. Model Preparation

**Goal:** Train and save XGBoost model for deployment.

**Why train on full dataset:**

In notebook 3, we used 80/20 split to EVALUATE XGBoost performance (99.995% recall achieved). 

Now for deployment, we retrain on ALL 235,795 URLs to give the model maximum training data. This is standard ML practice:
- **Evaluation phase**: Use split to test performance
- **Deployment phase**: Retrain on all data for strongest model

We are NOT re-testing - we accept the circular validation limitation acknowledged in notebook 3.

**Steps:**
1. Load dataset4 and prepare features (49 numeric features)
2. Train XGBoost on full dataset
3. Save model to `models/xgb_model.pkl`
4. Validate loading works

In [ ]:
import pandas as pd
import numpy as np
import pickle
import os
from xgboost import XGBClassifier

# Load dataset4
df = pd.read_csv('data/dataset4.csv')

print(f"Total URLs: {len(df):,}")
print(f"Phishing: {(df['label']==0).sum():,} ({(df['label']==0).sum()/len(df)*100:.1f}%)")
print(f"Legitimate: {(df['label']==1).sum():,} ({(df['label']==1).sum()/len(df)*100:.1f}%)")

# Prepare features (same as notebook 3)
features_to_exclude = ['URLSimilarityIndex', 'FILENAME', 'URL', 'Domain', 'TLD', 'Title', 'label']
feature_cols = [col for col in df.columns if col not in features_to_exclude]

X_full = df[feature_cols]
y_full = df['label']

print(f"\nUsing {len(feature_cols)} features for training")

In [ ]:
# Train XGBoost on full dataset
xgb_model = XGBClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss'
)

print("Training XGBoost on full dataset (235,795 URLs)...")
xgb_model.fit(X_full, y_full)
print("Training complete!")

# Create models directory if it doesn't exist
os.makedirs('models', exist_ok=True)

# Save model
model_path = 'models/xgb_model.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(xgb_model, f)

print(f"\nModel saved to: {model_path}")
print(f"Model file size: {os.path.getsize(model_path) / 1024 / 1024:.2f} MB")

In [ ]:
# Validate: Load model and test prediction
print("Validating model loading...")

with open(model_path, 'rb') as f:
    loaded_model = pickle.load(f)

# Test on first 5 URLs
test_sample = X_full.head(5)
predictions = loaded_model.predict(test_sample)
probabilities = loaded_model.predict_proba(test_sample)

print("\nTest predictions on first 5 URLs:")
for i in range(5):
    actual = "Phishing" if y_full.iloc[i] == 0 else "Legitimate"
    pred = "Phishing" if predictions[i] == 0 else "Legitimate"
    prob_phishing = probabilities[i][0] * 100
    prob_legitimate = probabilities[i][1] * 100
    
    print(f"URL {i+1}: Actual={actual}, Predicted={pred}, Prob(Phishing)={prob_phishing:.1f}%, Prob(Legitimate)={prob_legitimate:.1f}%")

print("\n✓ Model loading and prediction working correctly!")

## 3. Hybrid Production Architecture

**Why hybrid (Rules + ML) instead of ML-only?**

Even though XGBoost achieves 100% accuracy, we implement a hybrid approach for production:

1. **Speed**: Rules process 88.2% instantly (no ML computation needed)
2. **Explainability**: "Blocked: No HTTPS" vs "ML scored 95.3% phishing probability"
3. **Cost Efficiency**: Why run ML on ALL URLs when most can be filtered with simple if-statements?
4. **Reliability**: Rules provide fallback if ML model fails in production
5. **Best Practice**: Multi-stage filtering is standard (cheap filters first, expensive ML last)

**Production Flow:**

```
User submits URL
     ↓
┌─────────────────────────────────────┐
│ Stage 1: Rule-Based Filter (88.2%)  │
│ - Check 7 perfect rules             │
│ - Takes <1ms                        │
│ - If matched: BLOCK immediately     │
└─────────────────────────────────────┘
     ↓ (11.8% pass through)
┌─────────────────────────────────────┐
│ Stage 2: ML Scoring (11.8%)         │
│ - Load features (pre-extracted)     │
│ - XGBoost prediction                │
│ - Takes ~10-50ms                    │
│ - Output: Probability score         │
└─────────────────────────────────────┘
     ↓
┌─────────────────────────────────────┐
│ Decision Engine                     │
│ - HIGH risk (>90%): Flag for review │
│ - MEDIUM (70-90%): Manual review    │
│ - LOW (<70%): Allow                 │
└─────────────────────────────────────┘
```

**Concrete Examples:**

**Example 1: Easy phishing (88.2% of cases)**
- URL: `http://192.168.1.1/login.php`
- Stage 1: Rule 2 (No HTTPS) triggers
- Result: **BLOCKED** in <1ms
- Stage 2: Never reached (saved computation)

**Example 2: Sophisticated phishing (11.8% of cases)**
- URL: `https://www.seedjfly.top` (the ultimate edge case)
- Stage 1: All 7 rules pass (has HTTPS, resources, trust signals)
- Stage 2: XGBoost scores → 85% phishing probability
- Result: **FLAGGED for manual review** (~50ms total)

**Example 3: Legitimate site**
- URL: `https://google.com`
- Stage 1: All rules pass
- Stage 2: XGBoost scores → 0.1% phishing probability
- Result: **ALLOWED** (~50ms total)

**Performance at scale:**

If processing 10,000 URLs/day:
- 8,820 URLs: Blocked by rules (~8 seconds total)
- 1,180 URLs: Scored by ML (~60 seconds total)
- **Total: ~68 seconds for 10,000 URLs**

vs ML-only approach: ~500 seconds (8+ minutes)

**7x faster with hybrid approach!**

### 3.1 Rule Implementation

The 7 perfect rules from notebook 2, implemented as a production function:

In [ ]:
def apply_rules(row):
    """
    Apply 7 perfect rules from notebook 2.
    
    Returns:
        (caught, rule_name) tuple
        - caught: True if rule triggered (block immediately)
        - rule_name: Which rule caught it (for explanation)
    
    If caught=False, URL passes to Stage 2 (ML scoring).
    """
    
    # Rule 1: Zero Resources
    if row['NoOfJS'] == 0 and row['NoOfCSS'] == 0 and row['NoOfImage'] == 0:
        return (True, "Rule 1: Zero Resources")
    
    # Rule 2: No HTTPS
    if row['IsHTTPS'] == 0:
        return (True, "Rule 2: No HTTPS")
    
    # Rule 3: Domain is IP
    if row['IsDomainIP'] == 1:
        return (True, "Rule 3: Domain is IP")
    
    # Rule 4: Zero Trust Signals
    if (row['HasTitle'] == 0 and row['HasFavicon'] == 0 and 
        row['HasDescription'] == 0 and row['HasCopyrightInfo'] == 0):
        return (True, "Rule 4: Zero Trust Signals")
    
    # Rule 5: No References
    if (row['NoOfExternalRef'] == 0 and row['NoOfSelfRef'] == 0 and 
        row['NoOfEmptyRef'] == 0):
        return (True, "Rule 5: No References")
    
    # Rule 6: Many Subdomains
    if row['NoOfSubDomain'] >= 5:
        return (True, "Rule 6: NoOfSubDomain >= 5")
    
    # Rule 7: Long URL
    if row['URLLength'] > 57:
        return (True, "Rule 7: URLLength > 57")
    
    # No rule triggered - pass to ML
    return (False, None)


# Test the function on a few URLs
print("Testing rule function on sample URLs:\n")

for i in range(10):
    row = df.iloc[i]
    caught, rule = apply_rules(row)
    actual = "Phishing" if row['label'] == 0 else "Legitimate"
    
    if caught:
        print(f"URL {i+1}: {row['URL'][:50]}...")
        print(f"  Actual: {actual} | Caught by: {rule}\n")
    else:
        print(f"URL {i+1}: {row['URL'][:50]}...")
        print(f"  Actual: {actual} | Passed rules → needs ML scoring\n")

## 4. Streamlit Interactive Demo

**The complete deployment app is in `app.py`**

### 4.1 App Features

**Interactive URL Selection:**
- Filter by: All URLs / Phishing Only / Legitimate Only
- Dropdown selector with 235,795 URLs from dataset4
- Quick access to 5 edge cases (seedjfly.top, blogspot phishing, etc.)

**Hybrid Detection Flow:**

**Stage 1: Rule-Based Filter**
- Apply 7 perfect rules
- If caught: Show which rule triggered + explanation
- Display "BLOCKED" decision immediately

**Stage 2: ML Scoring (if rules passed)**
- XGBoost prediction with probability scores
- Risk assessment: HIGH (>90%), MEDIUM (70-90%), LOW (<70%)
- Recommendation: Block / Review / Allow
- Top contributing features display

**Additional Info:**
- Actual label comparison (test accuracy)
- System performance metrics
- Feature breakdown

### 4.2 Running the App

**Installation:**
```bash
pip install streamlit
```

**Launch:**
```bash
streamlit run app.py
```

**Access:**
- Browser opens automatically at `http://localhost:8501`
- Interactive demo ready to use

### 4.3 App Screenshots (Key Features)

1. **URL Selection:** Dropdown with filters (All/Phishing/Legitimate)
2. **Stage 1 Result:** "BLOCKED by Rule 2: No HTTPS" with explanation
3. **Stage 2 Result:** Risk meter showing 95.3% phishing probability
4. **Edge Case:** seedjfly.top showing 85% phishing (passed rules, caught by ML)
5. **Performance Stats:** 100% accuracy, 0 false positives

The app demonstrates the complete hybrid system in an interactive, stakeholder-friendly format.